In [ ]:
import cv2
import torch
import torchvision
import numpy as np
from PIL import Image
class PreprocessToMNIST:
    def __call__(self, img):
        img = img.convert("L")
        # normalize
        img = np.array(img).astype(np.float32) / 255.0

        # Invert colors if black on white
        if img.mean() > 0.5:
            img = 1 - img
        

        # Find white pixels, compute bound of number, and crop it to match MNIST 20x20 bounds
        mask = img > 0.1
        coords = np.argwhere(mask)
        if coords.size == 0:
            return Image.fromarray(np.zeros((28, 28), dtype=np.uint8))
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0)

        cropped = img[y0:y1+1, x0:x1+1]
        resized = cv2.resize(cropped, (20, 20), interpolation=cv2.INTER_AREA)

        # Pad 20x20
        new_img = np.zeros((28, 28), dtype=np.float32)
        new_img[4:24, 4:24] = resized

        return Image.fromarray((new_img * 255).astype(np.uint8))


print('normalize transform starting')
normalize_transform = torchvision.transforms.Compose([
    PreprocessToMNIST(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=(0.1307,), std=(0.3081,))
    ])

class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        
        self.model = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=1, out_channels=32,
                        kernel_size=3, padding='same'),
            torch.nn.BatchNorm2d(num_features=32),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=32, out_channels=32,
                        kernel_size=3, padding='same'),
            torch.nn.BatchNorm2d(num_features=32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2),
            torch.nn.Conv2d(in_channels=32, out_channels=64,
                        kernel_size=3, padding='same'),
            torch.nn.BatchNorm2d(num_features=64),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=64, out_channels=64,
                        kernel_size=3, padding='same'),
            torch.nn.BatchNorm2d(num_features=64),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2),
            torch.nn.Flatten(),
            torch.nn.LazyLinear(128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 10)
        )
    def forward(self,x):
        return self.model(x)